In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
# Import all 3 datasets
recalls_a = pd.read_csv("../data/csv/recalls_2016_2021.csv")
recalls_b = pd.read_csv("../data/csv/recalls_2022_2025.csv")
recalls_c = pd.read_csv("../data/csv/recalls_2025_april.csv")

In [17]:
# Subset 3 important columns, reasons
recalls_a_edit = recalls_a[["year", "product_name", "manufacturer","reason"]]
recalls_b_edit = recalls_b[["year", "product_name", "manufacturer","reason"]]
recalls_c_edit = recalls_c[["year", "product_name", "manufacturer","reason"]]

In [18]:
stacked_reason = pd.concat([recalls_a_edit, recalls_b_edit, recalls_c_edit], ignore_index=True)

In [19]:
# Function to categorize reasons based on keywords for grouping similar ones
def categorize(reason):
    r = reason.lower()
    if 'color' in r or 'colour' in r or 'discolor' in r or 'spot' in r or 'brown' in r or 'black' in r or 'dark' in r or 'mould' in r or 'mold' in r or 'crack' in r or 'crumb' in r or 'broken' in r or 'appearance' in r or 'coat' in r:
        return 'Appearance Issues'
    elif 'test' in r or 'oos' in r or 'out of' in r or 'fail' in r or 'comply' in r or 'stability' in r or 'specification' in r or 'trend' in r or 'friability' in r or 'uniformity' in r:
        return 'Quality Test Failures'
    elif 'voluntary' in r or 'withdrawall' in r:
        return 'Voluntary Recall'
    else:
        return 'Other'

# Apply categorization
stacked_reason['category'] = stacked_reason['reason'].apply(categorize)

# Group by category, count occurrences, sort descending, and get top 3
top_3_groups = stacked_reason.groupby('category').size().sort_values(ascending=False).head(3)

# Display the result
print(top_3_groups)

category
Quality Test Failures    71
Appearance Issues        51
Other                    38
dtype: int64


In [7]:
# Function to categorize reasons from a public health perspective
def public_health_categorize(reason):
    r = reason.lower()
    # Efficacy and Potency Risks: Focus on test failures related to strength, dissolution, stability
    if 'assay' in r or 'dissolution' in r or 'oos' in r or 'out of specification' in r or 'out of trend' in r or 'stability' in r or 'fail' in r or 'comply' in r or 'uniformity' in r or 'weight variation' in r or 'friability' in r or 'related substances' in r or 'content' in r or 'impurity' in r or 'thickness' in r or 'width' in r or 'holes' in r:
        return 'Efficacy and Potency Risks'
    # Safety and Contamination Risks: Visible defects, contamination indicators
    elif 'color' in r or 'colour' in r or 'discolor' in r or 'spot' in r or 'brown' in r or 'black' in r or 'dark' in r or 'mould' in r or 'mold' in r or 'particle' in r or 'particulate' in r or 'crystallization' in r or 'crack' in r or 'broken' in r or 'crumb' in r or 'precipitate' in r or 'corroded' in r or 'microbial' in r or 'moulds' in r or 'agglomerated' in r or 'lumping' in r or 'odor' in r or 'taste' in r or 'diethylene glycol' in r:
        return 'Safety and Contamination Risks'
    # Access and Compliance Risks: Labeling, packaging, voluntary recalls for those reasons
    elif 'voluntary' in r or 'withdrawall' in r or 'label' in r or 'pack' in r or 'mix-up' in r or 'mix up' in r or 'mislabeling' in r or 'incomplete' in r or 'leak' in r or 'leaking' in r or 'damage' in r or 'defect' in r or 'pinholes' in r or 'authorization' in r or 'capping' in r or 'lamination' in r:
        return 'Access and Compliance Risks'
    # Other/Emerging Risks: Miscellaneous
    else:
        return 'Other/Emerging Risks'

In [12]:
stacked_reason['public_health_category'] = stacked_reason['reason'].apply(public_health_categorize)
stacked_reason['drug_category'] = stacked_reason['product_name'].apply(categorize_product)

# Group by category, count occurrences, sort descending, and display
# Group by drug category and public health reason, count occurrences
result = stacked_reason.groupby(['drug_category', 'public_health_category']).size().unstack(fill_value=0)

In [13]:
# Add total recalls per drug category
result['Total Recalls'] = result.sum(axis=1)

# Find the most common public health reason per drug category
result['Most Common Reason'] = result[['Efficacy and Potency Risks', 'Safety and Contamination Risks', 'Access and Compliance Risks', 'Other/Emerging Risks']].idxmax(axis=1)

# Calculate percentages for the most common reason
result['Percentage'] = (result[['Efficacy and Potency Risks', 'Safety and Contamination Risks', 'Access and Compliance Risks', 'Other/Emerging Risks']].max(axis=1) / result['Total Recalls'] * 100).round(2)

In [14]:
def get_distribution(row):
    dist = []
    for col in ['Efficacy and Potency Risks', 'Safety and Contamination Risks', 'Access and Compliance Risks', 'Other/Emerging Risks']:
        if col != row['Most Common Reason'] and row[col] > 0:
            dist.append(f"{col}: {row[col]/row['Total Recalls']*100:.2f}%")
    return ', '.join(dist) if dist else 'None'

result['Other Distributions'] = result.apply(get_distribution, axis=1)

# Select relevant columns for output
summary = result[['Total Recalls', 'Most Common Reason', 'Percentage', 'Other Distributions']]

# Display the result
print(summary)

public_health_category      Total Recalls              Most Common Reason  \
drug_category                                                               
Analgesics/Pain Relievers               9  Safety and Contamination Risks   
Anthelmintics                           5      Efficacy and Potency Risks   
Anti-infectives (Other)                 3      Efficacy and Potency Risks   
Antibiotics/Antimicrobials             20      Efficacy and Potency Risks   
Antifungals                             4     Access and Compliance Risks   
Antiseptics/Topical Agents              6      Efficacy and Potency Risks   
Antivirals                              4      Efficacy and Potency Risks   
Cardiovascular Drugs                   12      Efficacy and Potency Risks   
Contraceptives                          4      Efficacy and Potency Risks   
Gastrointestinal Drugs                  8      Efficacy and Potency Risks   
Medical Devices                         5     Access and Compliance Risks   

In [9]:
print(category_counts)

public_health_category
Efficacy and Potency Risks        76
Safety and Contamination Risks    62
Access and Compliance Risks       20
Other/Emerging Risks              11
dtype: int64


In [10]:
# Function to categorize products into drug groups
def categorize_product(product):
    p = product.lower()
    # Analgesics/Pain Relievers
    if 'paracetamol' in p or 'painil' in p or 'curamol' in p or 'tamedol' in p or 'dolac' in p or 'dawasprin' in p or 'aspirin' in p or 'medimol' in p:
        return 'Analgesics/Pain Relievers'
    # Anthelmintics
    elif 'alben' in p or 'trogyl' in p or 'abz' in p or 'tanzol' in p or 'olworm' in p:
        return 'Anthelmintics'
    # Anti-infectives (Other, non-antibiotic/anti-fungal)
    elif 'rhoclone' in p or 'biodine' in p or 'oralex' in p:
        return 'Anti-infectives (Other)'
    # Antibiotics/Antimicrobials
    elif 'mox' in p or 'clav' in p or 'cip' in p or 'strox' in p or 'kemoxyl' in p or 'altacef' in p or 'metrowin' in p or 'biotrim' in p or 'cosatrim' in p or 'clindar' in p or 'azithro' in p or 'cephin' in p or 'dawaflox' in p or 'mesporin' in p:
        return 'Antibiotics/Antimicrobials'
    # Antifungals
    elif 'kenazole' in p or 'dazole' in p or 'fluconazole' in p or 'silvad' in p:
        return 'Antifungals'
    # Antiseptics/Topical Agents
    elif 'gel' in p or 'cream' in p or 'spray' in p or 'umbicare' in p or 'pharmasal' in p or 'diarim' in p or 'klincord' in p:
        return 'Antiseptics/Topical Agents'
    # Antivirals
    elif 'atazor' in p or 'tenofovir' in p:
        return 'Antivirals'
    # Cardiovascular Drugs
    elif 'acepril' in p or 'warfarin' in p or 'losar' in p or 'irbesatan' in p or 'frusemide' in p or 'carvedi' in p or 'flamodip' in p or 'medopress' in p or 'bisotrol' in p:
        return 'Cardiovascular Drugs'
    # Contraceptives
    elif 'condom' in p or 'levonogets' in p:
        return 'Contraceptives'
    # Gastrointestinal Drugs
    elif 'dawapraz' in p or 'rabemac' in p or 'magnocid' in p or 'stopacid' in p or 's-prazo' in p or 'alkar-ur' in p:
        return 'Gastrointestinal Drugs'
    # Medical Devices
    elif 'suture' in p or 'microplas' in p or 'vasofix' in p or 'stapler' in p:
        return 'Medical Devices'
    # Other (miscellaneous drugs, e.g., respiratory, antihistamines)
    elif 'triohist' in p or 'upacof' in p or 'toramin' in p or 'carbocysten' in p or 'trichohist' in p or 'denk-air' in p or 'shaltoux' in p or 'caryl' in p:
        return 'Respiratory Drugs'
    # Psychotropics
    elif 'fluphenazine' in p or 'chlorpromazine' in p or 'amitrip' in p:
        return 'Psychotropics'
    # Vitamins/Supplements
    elif 'ascorbic' in p or 'famcal' in p or 'dts-z' in p or 'mz-cal' in p or 'topfoliferum' in p:
        return 'Vitamins/Supplements'
    # Other Pharmaceuticals (miscellaneous)
    elif 'tablet' in p or 'suspension' in p or 'injection' in p or 'syrup' in p or 'zolex' in p or 'lamprene' in p or 'atracurium' in p or 'nurabucaine' in p or 'lidocel' in p or 'lumidol' in p or 'paragen' in p or 'blink injection' in p or 'metrosim' in p or 'medzol' in p or 'benylin' in p or 'normnil' in p or 'dopamac' in p or 'biodopa' in p or 'totomol' in p or 'wombit' in p or 'zenagent' in p or 'jotonol' in p or 'zegra' in p or 'ventil' in p or 'atravita' in p or 'diprofos' in p or 'phenysod' in p:
        return 'Other Pharmaceuticals'
    # Uncategorized
    else:
        return 'Uncategorized'

In [ ]:
df['drug_category'] = df['product_name'].apply(categorize_product)